# ExamForge AI — preprocessing, LoRA-only and SFT+LoRA (corrected, no DPO)

**Qwen3-1.7B | LoRA-only | SFT+LoRA**

Keep this notebook, `examforge_experiments.py` and `requirements.txt` in the same folder. No QLoRA, DPO, RAG, optimization or model-quality evaluation here.

**Fast experiments:** every training function accepts `max_train=N` (or `train_fraction=0.1`), `max_val=N` and `epochs=1`. The subset is balanced over SciQ / RACE / EduQG and is the same for every trial (fixed seed), so trials stay comparable.

**Long runs:** start them from a terminal with `nohup` (see the last section) so a browser disconnect does not stop them.

## 1. Install dependencies
Install a CUDA build of PyTorch first.

In [ ]:
%pip install -r requirements.txt

## 2. Check GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## 3. Check EduQG (no GPU, no HuggingFace)
Shows how many EduQG questions are kept per split and why others are removed. If the download fails, put `qg_train_v0.json` and `qg_valid_v0.json` in `data/raw/eduqg/`.

In [ ]:
from examforge_experiments import check_eduqg
check_eduqg()

## 4. Preprocess SciQ, RACE and EduQG
Stops with an error if one dataset produces no examples.

In [ ]:
from examforge_experiments import preprocess
preprocess()          # preprocess(test_per_source=100) for a less noisy test set

## 5. Inspect the preprocessing report
`by_source` must show **all three** datasets with non-zero train, validation and test counts.

In [ ]:
import json, pandas as pd
from pathlib import Path
report = json.loads(Path("data/processed/preprocessing_report.json").read_text())
display(pd.DataFrame(report["by_source"]).T)
report["counts"], report["removed"]

## 6. Check the label masking (tokenizer only, no GPU)
The model must be trained only on the JSON answer, never on the passage.

In [ ]:
from examforge_experiments import check_masking
check_masking()

## 7. Recommended order for training
1. **Smoke test** (about 2-3 min): `max_train=300, max_val=60, epochs=1` — checks that everything runs and saves.
2. **Pilot** (about 15 min): `max_train=3000, max_val=150, epochs=1` — first real loss curve.
3. **Trials** with the same subset, one change at a time; then a **final run** with a larger subset (`max_train=9000`) or the full set, 1 epoch.

## 8. SFT+LoRA (main model)
SFT teaches the passage-to-MCQ task (loss only on the JSON answer); LoRA trains only small adapters. Base model in bf16/fp16, not 4-bit.

In [ ]:
from examforge_experiments import run_sft_lora
# 1) smoke test
run_sft_lora("baseline", max_train=300, max_val=60, epochs=1)
# 2) pilot
# run_sft_lora("baseline", max_train=3000, max_val=150, epochs=1)
# 3) trials on the SAME subset, one hyperparameter at a time
# run_sft_lora("trial1_lr",       max_train=3000, max_val=150, epochs=1)
# run_sft_lora("trial2_dropout",  max_train=3000, max_val=150, epochs=1)
# run_sft_lora("trial3_capacity", max_train=3000, max_val=150, epochs=1)
# full data: run_sft_lora("baseline")

In [ ]:
from examforge_experiments import plot_training_curves
plot_training_curves("outputs/sft_lora/baseline")   # add more trial folders to compare

## 9. LoRA-only (control)
Trains the same kind of adapters on passage text only (no question examples). Its loss cannot be compared with the SFT loss.

In [ ]:
from examforge_experiments import run_lora_only
# run_lora_only("baseline", max_train=3000, max_val=150, epochs=1)
# run_lora_only("baseline", resume=True)     # continue an interrupted run from its last checkpoint

In [ ]:
from examforge_experiments import plot_training_curves
# plot_training_curves("outputs/lora_only/baseline")

## 10. Long runs from a terminal
```bash
nohup python examforge_experiments.py train-sft-lora --trial baseline --max-train 9000 --epochs 1 > sft_baseline.log 2>&1 &
tail -f sft_baseline.log
```

## 11. Where outputs are saved
- Preprocessing: `data/processed/` (EduQG raw files cached in `data/raw/eduqg/`)
- LoRA-only adapters: `outputs/lora_only/<trial>/best_adapter`
- SFT+LoRA adapters: `outputs/sft_lora/<trial>/best_adapter`
- Per run: `run_config.json` (hyperparameters and subset sizes per dataset) and `training_log.json` (loss history)